# CFD starter — D2Q9 Lattice-Boltzmann channel flow

Workflow:

1. Build a `ChannelFlow` spec (grid, Reynolds number).
2. Trace one LBM time step (BGK collision + streaming) as a matmul.
3. Preflight, read the Pareto table, pick a hardware plan.
4. Submit and read back the post-step state vector.
5. Sanity-check against the NumPy reference.

Required env vars: `UNIQX_API_KEY`, `UNIQX_GATEWAY`.

## 1 · Setup

In [ ]:
import os

import uniqx
from uniqx.cfd import ChannelFlow, build_lbm_step_module

GATEWAY = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
uniqx.login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY)
client = uniqx.connect(GATEWAY)
print("uniqx", uniqx.__version__, "— gateway:", GATEWAY)

## 2 · Workload — 64 × 16 channel at Re=100

In [ ]:
flow = ChannelFlow(nx=64, ny=16, Re=100.0)
print(f"grid: {flow.nx} x {flow.ny} | nu={flow.nu:.5f} | tau={flow.tau:.5f}")

module, runtime_inputs, meta = build_lbm_step_module(flow)
print(f"traced ops    : {len(module.functions[0].ops)}")
print(f"state size    : {meta['state_size']} (= dim * 9 lattice velocities)")

## 3 · Preflight

In [ ]:
options = uniqx.preflight(module, client=client)
print(options.summary())

In [ ]:
choice = options.recommended
print(f"Selected: {choice['label']}")
print(f"  node assignment : {choice.get('node_assignments', {})}")

## 4 · Submit & retrieve

In [ ]:
job_id = uniqx.submit(
    module,
    client=client,
    preflight_job_id=options.job_id,
    option_idx=choice["_idx"],
    runtime_inputs=runtime_inputs,
)
result = uniqx.get(job_id, client=client, timeout=300)

payload = result.get("payload") or result.get("result_payload") or b""
if isinstance(payload, str):
    payload = payload.encode("utf-8", errors="replace")

outputs = []
for line in payload.decode("utf-8", errors="replace").splitlines():
    parsed = uniqx.parse_buffer_view(line.strip())
    if parsed is not None:
        outputs.append(parsed)

# Module returns (f_new, energy)
f_new_dims, _, f_new_vals = outputs[0]
_, _, energy_vals = outputs[1]
print(f"f_new shape : {f_new_dims} ({len(f_new_vals)} values)")
print(f"kinetic E   : {energy_vals[0]:.6f}")

## 5 · Baseline — NumPy reference

In [ ]:
from baseline import initial_state, kinetic_energy, lbm_step

f = initial_state(flow.nx, flow.ny)
f = lbm_step(f, flow.tau, periodic_x=True)
ref_energy = kinetic_energy(f)

print(f"uniqx kinetic energy : {energy_vals[0]:.6f}")
print(f"NumPy reference      : {ref_energy:.6f}")
print(f"relative diff        : {abs(energy_vals[0] - ref_energy) / abs(ref_energy):.2e}")

## 6 · Discussion (your job)

1. **Scale to 256 × 64.** The operator matrix grows ~16×. Re-run preflight
   and paste both `summary()` tables side by side.
2. **Drop Re to 10.** Watch the `tau` knob and the recommended option.
3. **Compose multiple steps.** Submit a 100-step trajectory and compare
   the velocity profile at the outlet against the analytical Poiseuille
   parabola.